In [1]:
# Import pustaka untuk persiapan deployment dan pengujian server Flask
import os
import io
import shutil
from pathlib import Path
import cv2
import numpy as np
import pickle
import tensorflow as tf

In [2]:
# Verifikasi keberadaan artefak model day_night_model.h5 dan scaler.pkl
deploy_dir = 'daynight-project'
os.makedirs(deploy_dir, exist_ok=True)

model_path = os.path.join(deploy_dir, 'day_night_model.h5')
scaler_path = os.path.join(deploy_dir, 'scaler.pkl')

if not os.path.exists(model_path) and os.path.exists('day_night_model.h5'):
    shutil.copy2('day_night_model.h5', model_path)
if not os.path.exists(scaler_path) and os.path.exists('scaler.pkl'):
    shutil.copy2('scaler.pkl', scaler_path)

print(f'File model tersedia : {os.path.exists(model_path)}')
print(f'File scaler tersedia: {os.path.exists(scaler_path)}')

File model tersedia : True
File scaler tersedia: True


In [3]:
# Pembuatan file aplikasi server web Flask app.py
app_code = '''import os
import numpy as np
import cv2
import pickle
import tensorflow as tf
from flask import Flask, request, render_template_string
from skimage.feature import hog

app = Flask(__name__)

# Load Model & Scaler
MODEL_PATH = 'day_night_model.h5'
SCALER_PATH = 'scaler.pkl'

try:
    model = tf.keras.models.load_model(MODEL_PATH)
    with open(SCALER_PATH, 'rb') as f:
        scaler = pickle.load(f)
    print("System Loaded Successfully")
except Exception as e:
    print(f"Error loading system: {e}")

def preprocess_image(image_bytes):
    nparr = np.frombuffer(image_bytes, np.uint8)
    img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
    img = cv2.resize(img, (256, 256))
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    hog_feat = hog(
        gray, orientations=9, pixels_per_cell=(8,8),
        cells_per_block=(2,2), block_norm='L2-Hys',
        visualize=False, feature_vector=True
    )
    return scaler.transform(hog_feat.reshape(1, -1))

@app.route('/', methods=['GET'])
def home():
    return render_template_string(\'\'\'
    <!DOCTYPE html>
    <html lang="id">
    <head>
        <meta charset="UTF-8">
        <title>Day vs Night Classifier</title>
        <style>
            body { font-family: Arial, sans-serif; background-color: #f4f6f9; display: flex; justify-content: center; align-items: center; min-height: 80vh; }
            .card { background: white; padding: 40px; border-radius: 12px; box-shadow: 0 4px 12px rgba(0,0,0,0.1); text-align: center; max-width: 450px; width: 100%; }
            h1 { color: #2c3e50; font-size: 24px; margin-bottom: 20px; }
            input[type="file"] { margin: 20px 0; padding: 10px; border: 1px dashed #3498db; border-radius: 6px; width: 80%; }
            button { background-color: #3498db; color: white; border: none; padding: 12px 24px; font-size: 16px; border-radius: 6px; cursor: pointer; font-weight: 600; }
            button:hover { background-color: #2980b9; }
        </style>
    </head>
    <body>
        <div class="card">
            <h1>Day vs Night Classifier</h1>
            <p>Unggah citra untuk mengklasifikasikan siang atau malam.</p>
            <form action="/predict" method="post" enctype="multipart/form-data">
                <input type="file" name="file" accept="image/*" required><br>
                <button type="submit">Prediksi</button>
            </form>
        </div>
    </body>
    </html>
    \'\'\')

@app.route('/predict', methods=['POST'])
def predict():
    try:
        file = request.files['file']
        data = preprocess_image(file.read())
        prediction = float(model.predict(data, verbose=0)[0][0])
        
        is_day = prediction > 0.5
        label = "Day (Siang)" if is_day else "Night (Malam)"
        confidence = prediction if is_day else (1.0 - prediction)
        badge_color = "#27ae60" if is_day else "#2c3e50"
        
        return f\'\'\'
        <!DOCTYPE html>
        <html lang="id">
        <head>
            <meta charset="UTF-8">
            <title>Hasil Prediksi</title>
            <style>
                body {{ font-family: Arial, sans-serif; background-color: #f4f6f9; display: flex; justify-content: center; align-items: center; min-height: 80vh; }}
                .card {{ background: white; padding: 40px; border-radius: 12px; box-shadow: 0 4px 12px rgba(0,0,0,0.1); text-align: center; max-width: 450px; width: 100%; }}
                .badge {{ display: inline-block; padding: 8px 18px; border-radius: 20px; color: white; background: {badge_color}; font-size: 20px; font-weight: bold; margin: 15px 0; }}
                a {{ display: inline-block; margin-top: 20px; color: #3498db; text-decoration: none; font-weight: 600; }}
            </style>
        </head>
        <body>
            <div class="card">
                <h2>Hasil Klasifikasi</h2>
                <div class="badge">{label}</div>
                <p>Confidence: {confidence*100:.2f}%</p>
                <p>Sigmoid: {prediction:.4f}</p>
                <a href="/">Kembali</a>
            </div>
        </body>
        </html>
        \'\'\'
    except Exception as e:
        return f"Error: {e}"

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=7860)
'''

with open(os.path.join(deploy_dir, 'app.py'), 'w', encoding='utf-8') as f:
    f.write(app_code)
print('File app.py berhasil ditulis di daynight-project/app.py')

File app.py berhasil ditulis di daynight-project/app.py


In [4]:
# Pembuatan file requirements.txt untuk container deployment
requirements_content = '''flask
tensorflow-cpu
numpy
scikit-learn
scikit-image
opencv-python-headless
gunicorn
'''

with open(os.path.join(deploy_dir, 'requirements.txt'), 'w', encoding='utf-8') as f:
    f.write(requirements_content)
print('File requirements.txt berhasil ditulis di daynight-project/requirements.txt')

File requirements.txt berhasil ditulis di daynight-project/requirements.txt


In [5]:
# Pembuatan konfigurasi Dockerfile untuk Hugging Face Spaces
dockerfile_content = '''# Base Image Python 3.9 Slim
FROM python:3.9-slim

# Install Library Sistem untuk OpenCV
RUN apt-get update && apt-get install -y \\
    libgl1 \\
    libglib2.0-0 \\
    && rm -rf /var/lib/apt/lists/*

# Setup User Non-Root (Standar Keamanan HF)
RUN useradd -m -u 1000 user
USER user
ENV PATH="/home/user/.local/bin:$PATH"

# Setup Direktori Kerja
WORKDIR /app

# Install Dependencies
COPY --chown=user ./requirements.txt requirements.txt
RUN pip install --no-cache-dir --upgrade -r requirements.txt

# Copy File Aplikasi
COPY --chown=user . /app

# Expose Port & Jalankan
EXPOSE 7860
CMD ["python", "app.py"]
'''

with open(os.path.join(deploy_dir, 'Dockerfile'), 'w', encoding='utf-8') as f:
    f.write(dockerfile_content)
print('File Dockerfile berhasil ditulis di daynight-project/Dockerfile')

File Dockerfile berhasil ditulis di daynight-project/Dockerfile


In [6]:
# Pengujian inferensi web app secara lokal menggunakan Flask test_client
import sys
import importlib.util

spec = importlib.util.spec_from_file_location('app', os.path.join(deploy_dir, 'app.py'))
flask_app_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(flask_app_module)

client = flask_app_module.app.test_client()

# Uji endpoint GET /
res_home = client.get('/')
print(f'Test GET / Status Code: {res_home.status_code}')
assert res_home.status_code == 200

# Uji endpoint POST /predict dengan sampel citra siang
sample_day = 'images/test/day/20151101_152050.jpg'
with open(sample_day, 'rb') as f:
    img_data_day = f.read()
res_day = client.post('/predict', data={'file': (io.BytesIO(img_data_day), 'day.jpg')}, content_type='multipart/form-data')
print(f'Test POST /predict (Day) Status: {res_day.status_code}')
print('Output Prediksi Siang:', 'Day (Siang)' in res_day.get_data(as_text=True))

# Uji endpoint POST /predict dengan sampel citra malam
sample_night = 'images/test/night/20151101_072507.jpg'
with open(sample_night, 'rb') as f:
    img_data_night = f.read()
res_night = client.post('/predict', data={'file': (io.BytesIO(img_data_night), 'night.jpg')}, content_type='multipart/form-data')
print(f'Test POST /predict (Night) Status: {res_night.status_code}')
print('Output Prediksi Malam:', 'Night (Malam)' in res_night.get_data(as_text=True))

System Loaded Successfully
Test GET / Status Code: 200
Test POST /predict (Day) Status: 200
Output Prediksi Siang: True


Test POST /predict (Night) Status: 200
Output Prediksi Malam: True


In [7]:
# Instruksi proses push repository ke Hugging Face Spaces via Git CLI
hf_guide = '''
Langkah-langkah Deployment ke Hugging Face Spaces:
1. Buat Space baru pada akun Hugging Face:
   - Space SDK : Docker
   - Visibility: Public
2. Buat Personal Access Token tipe 'Write' di Settings > Access Tokens.
3. Lakukan clone repository Space kosong ke lokal:
   git clone https://huggingface.co/spaces/<USERNAME>/<SPACE_NAME>
4. Salin seluruh isi folder daynight-project ke folder repository clone.
5. Lakukan commit dan push via terminal:
   cd <SPACE_NAME>
   git add .
   git commit -m "Deploy Day vs Night ML Classifier"
   git push
6. Masukkan Access Token sebagai password saat diminta otentikasi.
7. Tunggu build Docker selesai hingga status berubah menjadi Running.
'''
print(hf_guide)


Langkah-langkah Deployment ke Hugging Face Spaces:
1. Buat Space baru pada akun Hugging Face:
   - Space SDK : Docker
   - Visibility: Public
2. Buat Personal Access Token tipe 'Write' di Settings > Access Tokens.
3. Lakukan clone repository Space kosong ke lokal:
   git clone https://huggingface.co/spaces/<USERNAME>/<SPACE_NAME>
4. Salin seluruh isi folder daynight-project ke folder repository clone.
5. Lakukan commit dan push via terminal:
   cd <SPACE_NAME>
   git add .
   git commit -m "Deploy Day vs Night ML Classifier"
   git push
6. Masukkan Access Token sebagai password saat diminta otentikasi.
7. Tunggu build Docker selesai hingga status berubah menjadi Running.

